# End-to-end pipeline evaluation: retrieval -> ML selection

This tests the architecture you actually want:

```
offer  ->  RETRIEVER picks top-K SKUs from all 237
       ->  19 runtime features built for each candidate
       ->  LightGBM scores them
       ->  best candidate wins (or NO_MATCH below threshold)
```

Three retrievers are compared under an identical ML selection stage:

| Retriever | What it is |
|---|---|
| **RapidFuzz** | what production does today |
| **Embedding** | what you are proposing |
| **Union** | both, candidates merged |

You get two levels of metric:

1. **recall@K** - does the correct SKU even reach the shortlist? This is the
   hard ceiling on everything downstream; ML cannot pick what was never retrieved.
2. **end-to-end precision / recall / F1** - after LightGBM makes the final call.

The comparison of row 2 across the three retrievers is the number that settles
whether the embedding belongs beside RapidFuzz.

## 1. FILES YOU NEED TO PROVIDE

Put these in one folder on Google Drive named **`promostrater`** (or set
`PROMOSTRATER_DIR` if running locally). Keep the sub-folder structure:

```
promostrater/
  src/                                   <- the whole src folder from the repo
      sku_mapping/...                       (gives the EXACT runtime rules)
  models/
      alkabeer_sku_matcher_v1.joblib     <- the trained LightGBM bundle
  data/processed/
      test_split.parquet                 <- held-out gold pairs
      training_features.parquet          <- optional, for the larger gold set
  Product_Master.xlsx                    <- the 237-SKU catalogue
```

Easiest way: zip the repo, upload the zip to Drive, and unzip it there. Cell 2
checks every path and tells you exactly what is missing before anything runs.

In [ ]:
!pip -q install sentence-transformers rapidfuzz lightgbm joblib openpyxl pyarrow scikit-learn 2>/dev/null
import os, sys, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    import torch; DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'
print('device:', DEVICE)

## 2. Locate files (fails loudly if anything is missing)

In [ ]:
NEEDS_SRC = True

try:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/promostrater'
except Exception:
    BASE = os.environ.get('PROMOSTRATER_DIR', '.')
print('BASE =', BASE)

import glob
def find_file(fname):
    """Find a file ANYWHERE under BASE - the folder layout does not matter."""
    hits = [p for p in glob.glob(f'{BASE}/**/{fname}', recursive=True)
            if os.path.isfile(p)]
    hits.sort(key=len)
    return hits[0] if hits else None

def find_src():
    hits = glob.glob(f'{BASE}/**/sku_mapping/__init__.py', recursive=True)
    hits.sort(key=len)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None

MODEL_PATH  = find_file('alkabeer_sku_matcher_v1.joblib')
MASTER_PATH = find_file('Product_Master.xlsx')
TEST_PATH   = find_file('test_split.parquet')
VAL_PATH    = find_file('validation_split.parquet')
ALL_PATH    = find_file('training_features.parquet')
SRC_DIR     = find_src()

required = {'LightGBM model': MODEL_PATH, 'Product Master': MASTER_PATH,
            'test split': TEST_PATH}
if NEEDS_SRC:
    required['src package'] = SRC_DIR
missing = [k for k, v in required.items() if not v]
for k, v in required.items():
    print(f"{'OK     ' if v else 'MISSING'}  {k:15s} {v or '-- not found'}")
for k, v in (('validation split', VAL_PATH), ('all gold pairs', ALL_PATH)):
    print(f"{'OK     ' if v else 'absent '}  {k:15s} {v or '(optional)'}")

if missing:
    print(f'\n--- what IS present under {BASE} ---')
    seen = sorted(glob.glob(f'{BASE}/**/*', recursive=True))
    for p in seen[:50]:
        print('   ', p)
    if not seen:
        print('    (nothing - is the folder name exactly "promostrater"?)')
    raise FileNotFoundError('Could not find: ' + ', '.join(missing))

sys.path.insert(0, SRC_DIR)
from sku_mapping.features.feature_generator import build_feature_vector
from sku_mapping.features.measurement_features import (
    extract_flyer_measures, extract_master_measures)
from sku_mapping.features.text_features import clean_offer_text
print('\nimported the repo runtime rules - features match production exactly')


## 3. Load catalogue, model, and gold pairs

In [ ]:
import re, joblib

USE_ALL_GOLD = False   # False = held-out test split (honest), True = all positives

mst = pd.read_excel(MASTER_PATH)
for c in ['Itemname', 'Item-Cat-2', 'Item-Cat-4', 'Item Description', 'Item-Spec']:
    mst[c] = mst[c].fillna('')
mst['Itemcode'] = (mst['Itemcode'].fillna('').astype(str)
                   .str.replace(r'\.0$', '', regex=True).str.strip())
mst = mst.drop_duplicates('Itemcode').reset_index(drop=True)
mst['match_text'] = (mst['Itemname'] + ' ' + mst['Item-Cat-4'] + ' '
                     + mst['Item Description']).str.lower()\
                    .str.replace(r'\s+', ' ', regex=True).str.strip()
mst['measures_detailed'] = mst['Item-Spec'].apply(extract_master_measures)
N_SKU = len(mst)
codes = mst['Itemcode'].to_numpy()
code2row = {c: i for i, c in enumerate(codes)}
print(f'catalogue: {N_SKU} SKUs')

bundle = joblib.load(MODEL_PATH)
CLF, FEATS = bundle['model'], list(bundle['feature_columns'])
ML_T = float(bundle.get('auto_match_threshold', 0.5))
print(f'LightGBM loaded  |  {len(FEATS)} features  |  auto threshold {ML_T}')

src = ALL_PATH if (USE_ALL_GOLD and ALL_PATH) else TEST_PATH
gp = pd.read_parquet(src)
gp = gp[gp.pair_label == 1][['offer_text', 'master_itemcode']].dropna()
gp['master_itemcode'] = gp['master_itemcode'].astype(str)
gp = gp[gp.master_itemcode.isin(code2row)].drop_duplicates('offer_text')
gp = gp.reset_index(drop=True)
OFFERS = gp['offer_text'].astype(str).tolist()
GOLD = np.array([code2row[c] for c in gp['master_itemcode']])
print(f'evaluation offers: {len(OFFERS)}  (source: {os.path.basename(src)})')
print('NOTE: one gold SKU per offer; each offer is a single retrieval query.')

## 4. Retriever A - RapidFuzz (production today)

Same scorer the production candidate generator uses:
`(token_sort_ratio + token_set_ratio) / 2`.

In [ ]:
from rapidfuzz import fuzz, process
off_match = [re.sub(r'\s+', ' ', clean_offer_text(t)).strip() for t in OFFERS]
cat_match = mst['match_text'].tolist()
S_FUZZ = ((process.cdist(off_match, cat_match, scorer=fuzz.token_sort_ratio, workers=-1)
         + process.cdist(off_match, cat_match, scorer=fuzz.token_set_ratio, workers=-1))
         / 2.0).astype(np.float32)
print('RapidFuzz score matrix:', S_FUZZ.shape)

## 5. Retriever B - Embedding

In [ ]:
from sentence_transformers import SentenceTransformer
EMB_NAME = 'BAAI/bge-small-en-v1.5'
emb = SentenceTransformer(EMB_NAME, device=DEVICE)
def l2(a): return a / np.clip(np.linalg.norm(a, axis=1, keepdims=True), 1e-12, None)
V_OFF = l2(emb.encode(OFFERS, batch_size=64, convert_to_numpy=True, show_progress_bar=True))
V_CAT = l2(emb.encode(mst['match_text'].tolist(), batch_size=64,
                      convert_to_numpy=True, show_progress_bar=True))
S_EMB = (V_OFF @ V_CAT.T).astype(np.float32)
print('Embedding score matrix:', S_EMB.shape)

## 6. Retrieval quality - recall@K (the ceiling)

If the gold SKU is not in the top-K, no amount of ML skill can recover it.

In [ ]:
KS = (1, 3, 5, 10, 20)
def topk(S, k): return np.argsort(-S, axis=1)[:, :k]

rows = []
for k in KS:
    f_top, e_top = topk(S_FUZZ, k), topk(S_EMB, k)
    f_hit = (f_top == GOLD[:, None]).any(1)
    e_hit = (e_top == GOLD[:, None]).any(1)
    u_hit = np.array([GOLD[i] in set(f_top[i]) | set(e_top[i])
                      for i in range(len(GOLD))])
    rows.append({'K': k, 'fuzzy': f_hit.mean(), 'embedding': e_hit.mean(),
                 'union': u_hit.mean(), 'union_gain': u_hit.mean() - f_hit.mean(),
                 'fuzzy_misses': int((~f_hit).sum()),
                 'recovered_by_emb': int((~f_hit & e_hit).sum())})
rec = pd.DataFrame(rows)
print('=' * 72); print('RECALL@K  - does the correct SKU reach the shortlist?'); print('=' * 72)
print(rec.round(4).to_string(index=False))
print('\nunion_gain is the decision number: how much shortlist coverage the')
print('embedding ADDS on top of RapidFuzz. Near zero => not worth wiring in.')
r10 = rec[rec.K == 10].iloc[0]
print(f"\nat K=10: fuzzy {r10.fuzzy:.3f} | embedding {r10.embedding:.3f} | "
      f"union {r10.union:.3f}  (+{r10.union_gain:.3f})")
print(f"of {int(r10.fuzzy_misses)} offers fuzzy misses, the embedding recovers "
      f"{int(r10.recovered_by_emb)}")

## 7. ML selection stage

For each retriever: take its top-K, build the real 19 runtime features for every
candidate, score with LightGBM, and keep the best one above threshold.

In [ ]:
K_RETRIEVE = 10

master_rows = [{'Itemname': r['Itemname'], 'Item-Cat-4': r['Item-Cat-4'],
                'Item Description': r['Item Description'], 'Item-Spec': r['Item-Spec'],
                'master_measures_detailed': r['measures_detailed']}
               for _, r in mst.iterrows()]
offer_rows = [{'Offer Name': t, 'Product': '', 'Variant': '', 'Base Packsize': '',
               'offer_measures_detailed': extract_flyer_measures(' ' + t)}
              for t in OFFERS]

def run_pipeline(cand_lists, tag):
    """cand_lists[i] = candidate SKU row-indices for offer i."""
    feats, owner = [], []
    for i, cands in enumerate(cand_lists):
        for j in cands:
            feats.append(build_feature_vector(offer_rows[i], master_rows[j]))
            owner.append((i, j))
    F = pd.DataFrame(feats)[FEATS].astype(float).fillna(-1)
    p = (CLF.predict_proba(F)[:, 1] if hasattr(CLF, 'predict_proba')
         else np.asarray(CLF.predict(F), float))
    best = {}
    for (i, j), s in zip(owner, p):
        if i not in best or s > best[i][1]:
            best[i] = (j, float(s))
    pred = np.full(len(OFFERS), -1)
    conf = np.zeros(len(OFFERS))
    for i, (j, s) in best.items():
        conf[i] = s
        if s >= ML_T:
            pred[i] = j
    print(f'  {tag}: scored {len(p):,} candidate pairs')
    return pred, conf

f_top = topk(S_FUZZ, K_RETRIEVE); e_top = topk(S_EMB, K_RETRIEVE)
u_top = [sorted(set(f_top[i]) | set(e_top[i])) for i in range(len(OFFERS))]
print(f'building features and scoring (K={K_RETRIEVE}) ...')
P_F, C_F = run_pipeline(list(f_top), 'fuzzy    -> ML')
P_E, C_E = run_pipeline(list(e_top), 'embedding-> ML')
P_U, C_U = run_pipeline(u_top,       'union    -> ML')

## 8. End-to-end precision / recall / F1

In [ ]:
def e2e(pred, name, k):
    answered = pred >= 0
    correct  = answered & (pred == GOLD)
    tp = int(correct.sum())
    fp = int((answered & ~correct).sum())
    fn = int((~answered).sum())
    prec = tp / max(int(answered.sum()), 1)
    rec_ = tp / len(GOLD)
    f1   = 2 * prec * rec_ / (prec + rec_) if (prec + rec_) else 0.0
    return {'pipeline': name, 'K': k, 'precision': prec, 'recall': rec_, 'f1': f1,
            'correct': tp, 'wrong_sku': fp, 'no_answer': fn,
            'answered_%': answered.mean()}

summary = pd.DataFrame([
    e2e(P_F, 'RapidFuzz -> LightGBM  (production today)', K_RETRIEVE),
    e2e(P_E, 'Embedding -> LightGBM  (your proposal)',    K_RETRIEVE),
    e2e(P_U, 'Union     -> LightGBM  (both retrievers)',  K_RETRIEVE),
])
print('=' * 88)
print(f'END-TO-END  (retrieve top-{K_RETRIEVE} -> 19 runtime features -> '
      f'LightGBM >= {ML_T})')
print('=' * 88)
print(summary.round(4).to_string(index=False))
print('\nprecision = of the offers answered, how many got the RIGHT SKU')
print('recall    = of all offers, how many got the right SKU')
print('no_answer = every candidate fell below the ML threshold -> manual review')

b = summary.sort_values('f1', ascending=False).iloc[0]
print(f"\nbest pipeline by F1: {b.pipeline}  (F1 {b.f1:.3f})")
d = (summary.set_index('pipeline').f1.iloc[1] - summary.set_index('pipeline').f1.iloc[0])
print(f"embedding-vs-fuzzy F1 difference: {d:+.4f}")
print(f"union-vs-fuzzy    F1 difference: "
      f"{summary.f1.iloc[2] - summary.f1.iloc[0]:+.4f}   <-- the number that decides")

## 9. Sweep K, and see where the pipelines disagree

In [ ]:
sweep = []
for k in (5, 10, 20):
    ft, et = topk(S_FUZZ, k), topk(S_EMB, k)
    ut = [sorted(set(ft[i]) | set(et[i])) for i in range(len(OFFERS))]
    for nm, cl in (('fuzzy', list(ft)), ('embedding', list(et)), ('union', ut)):
        p, _ = run_pipeline(cl, f'{nm} K={k}')
        sweep.append(e2e(p, nm, k))
sw = pd.DataFrame(sweep)
print('\n' + sw.pivot(index='K', columns='pipeline',
                      values=['precision', 'recall', 'f1']).round(3).to_string())

det = pd.DataFrame({'offer_text': OFFERS,
                    'gold': codes[GOLD],
                    'fuzzy_pred': [codes[i] if i >= 0 else 'NO_MATCH' for i in P_F],
                    'emb_pred':   [codes[i] if i >= 0 else 'NO_MATCH' for i in P_E],
                    'union_pred': [codes[i] if i >= 0 else 'NO_MATCH' for i in P_U],
                    'fuzzy_conf': C_F.round(3), 'emb_conf': C_E.round(3)})
det['fuzzy_ok'] = det.fuzzy_pred == det.gold
det['emb_ok']   = det.emb_pred == det.gold
print('\nonly embedding correct:', int((det.emb_ok & ~det.fuzzy_ok).sum()),
      '| only fuzzy correct:', int((det.fuzzy_ok & ~det.emb_ok).sum()))
print('\noffers ONLY the embedding pipeline got right:')
print(det[det.emb_ok & ~det.fuzzy_ok][['offer_text', 'gold', 'fuzzy_pred']]
      .head(12).to_string(index=False))
det.to_csv('pipeline_predictions.csv', index=False)
summary.to_csv('pipeline_summary.csv', index=False)
print('\nwrote pipeline_predictions.csv, pipeline_summary.csv')

## 10. How to read the result

- **§6 recall@K** is the ceiling. If `union_gain` is ~0, the embedding finds
  nothing RapidFuzz missed and the whole idea is dead regardless of §8.
- **§8 end-to-end F1** is the payoff. Compare the three rows:
  - union > fuzzy  -> add the embedding as a second retriever (your plan works)
  - embedding > fuzzy -> consider replacing the retriever outright
  - both ~= fuzzy  -> keep the pipeline as it is
- **Union costs more compute** (more candidates to score), so a gain of a few
  tenths of a percent is not worth the latency. Judge the size, not the sign.

One honest caveat: the gold labels here come from the auto-labelled pool, part
of which is known to be wrong. A pipeline is penalised for "missing" a SKU whose
label was mistaken in the first place, so treat these as relative comparisons
between the three retrievers, not as absolute accuracy.